#### =============================================================================
#### SportsMOT on TeamTrack — UPGRADED VERSION
#### CSE 429 · Computer Vision and Pattern Recognition
#### =============================================================================
#### Fixes applied:
####   1. ByteTrack vs SORT — meaningful confidence split (0.35 / 0.15)
####   2. Ball trajectory  — Roboflow fine-tuned model + Kalman gate filter
####   3. Offside line     — homography + second-last defender rule
####   4. Defensive triangle — last 3 defenders convex overlay
####   5. Player overlap   — OSNet-AIN Re-ID alongside IoU in cost matrix
####   6. Team clustering  — per-ID centroid (non-overlapping frames only)
####   7. Local run fixes  — path constants, no ngrok, TrackEval pointer
#### =============================================================================

In [1]:
# ────────────────────────────────────────────────────────────────────────────────
# CELL 1 — Environment Setup (run once, then restart kernel)
# ────────────────────────────────────────────────────────────────────────────────
# FIRST RUN ONLY — uncomment and execute, then restart kernel:
#
# !pip install -q -U numpy scipy scikit-learn
# !pip install -q ultralytics supervision lapjv sahi filterpy
# !pip install -q git+https://github.com/roboflow/sports.git
# !pip install -q torchreid          # OSNet Re-ID
# !pip install -q imageio[ffmpeg]
#
# After restart, run everything from section 1.2 downward.

# ── 1.2  Imports ─────────────────────────────────────────────────────
import os, warnings, time, configparser
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, normalize
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from filterpy.kalman import KalmanFilter

from ultralytics import YOLO
import supervision as sv
import torch

# ── 1.3  Device ─────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── 1.4  Output directories ─────────────────────────────────────────────────
# ┌─ LOCAL RUN: change this to your local dataset root ────────────────────────────────────────
# │  DATASET_ROOT = Path("/path/to/your/teamtrack")                          │
# └───────────────────────────────────────────────────────────────────┘
DATASET_ROOT  = Path("/kaggle/input/datasets/atomscott/teamtrack")   # Kaggle
# DATASET_ROOT  = Path("./data/teamtrack")                           # Local

OUTPUT_DIR    = Path("outputs")
VIDEO_OUT_DIR = OUTPUT_DIR / "videos"
STATS_OUT_DIR = OUTPUT_DIR / "stats"
HEAT_OUT_DIR  = OUTPUT_DIR / "heatmaps"

for d in [VIDEO_OUT_DIR, STATS_OUT_DIR, HEAT_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output dirs ready: {OUTPUT_DIR}")
print("Cell 1 complete.")

Device : cuda
PyTorch: 2.10.0+cu128
GPU    : Tesla T4
Output dirs ready: outputs
Cell 1 complete.


In [29]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 2 — Dataset Exploration & Clip Selection  (unchanged from original)
# ────────────────────────────────────────────────────────────────────────────────
MOT_ROOT = DATASET_ROOT / "teamtrack-mot" / "teamtrack-mot"

all_sequences = []

for sv_dir in sorted(MOT_ROOT.iterdir()):
    if not sv_dir.is_dir():
        continue
    parts = sv_dir.name.split("_")
    view  = parts[-1] if parts[-1] in ("side", "top") else "unknown"
    sport = "_".join(parts[:-1]) if view != "unknown" else sv_dir.name
    print(f"\n[{sv_dir.name}]  sport={sport}  view={view}")

    for split_dir in sorted(sv_dir.iterdir()):
        if not split_dir.is_dir():
            continue
        seqs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
        print(f"  {split_dir.name}/  ({len(seqs)} sequences)")
        for seq in seqs:
            video_file = seq / "img1.mp4"
            has_video  = video_file.exists()
            has_gt     = (seq / "gt" / "gt.txt").exists()
            n_frames   = 0
            if has_video:
                cap      = cv2.VideoCapture(str(video_file))
                n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
            print(f"      {seq.name:<42}  frames:{n_frames:>4}  gt:{has_gt}  mp4:{has_video}")
            all_sequences.append({
                "sport": sport, "view": view, "split": split_dir.name,
                "name": seq.name, "path": seq, "video_file": video_file,
                "n_frames": n_frames, "has_video": has_video, "has_gt": has_gt,
            })

def pick_sequence(sequences, sport="soccer", split="train"):
    for filt in [
        lambda s: s["sport"]==sport and s["split"]==split and s["view"]=="side" and s["has_gt"] and s["has_video"],
        lambda s: s["sport"]==sport and s["split"]==split and s["has_gt"] and s["has_video"],
        lambda s: s["sport"]==sport and s["has_video"],
        lambda s: s["has_video"],
    ]:
        cands = sorted([s for s in sequences if filt(s)], key=lambda s: s["n_frames"])
        if cands:
            return cands[0]
    return None

chosen = pick_sequence(all_sequences)
if chosen is None:
    raise RuntimeError(f"No usable sequences under {MOT_ROOT}")

SELECTED_SEQ = chosen["path"]
VIDEO_FILE   = chosen["video_file"]
VIEW_TYPE    = chosen["view"]
GT_FILE      = SELECTED_SEQ / "gt" / "gt.txt"
SEQINFO_FILE = SELECTED_SEQ / "seqinfo.ini"

if SEQINFO_FILE.exists():
    cfg = configparser.ConfigParser()
    cfg.read(SEQINFO_FILE)
    si         = cfg["Sequence"]
    FPS        = int(float(si.get("frameRate", "25")))
    IMG_WIDTH  = int(float(si.get("imWidth",   "1920")))
    IMG_HEIGHT = int(float(si.get("imHeight",  "1080")))
    SEQ_LENGTH = int(float(si.get("seqLength", str(chosen["n_frames"]))))
else:
    cap        = cv2.VideoCapture(str(VIDEO_FILE))
    FPS        = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    IMG_WIDTH  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or 1920
    IMG_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 1080
    SEQ_LENGTH = chosen["n_frames"]
    cap.release()

ASPECT_RATIO = IMG_WIDTH / IMG_HEIGHT
IS_PANORAMIC = ASPECT_RATIO > 3.0

if IS_PANORAMIC:
    PIPELINE_MODE  = "panoramic"
    DET_CONF       = 0.15
    DET_IOU        = 0.45
    USE_SAHI       = True
    SAHI_SLICE_H   = 540
    SAHI_SLICE_W   = 540
    SAHI_OVERLAP   = 0.2
else:
    PIPELINE_MODE  = "broadcast"
    DET_CONF       = 0.35
    DET_IOU        = 0.45
    USE_SAHI       = False
    SAHI_SLICE_H   = None
    SAHI_SLICE_W   = None
    SAHI_OVERLAP   = None

print(f"\nSelected : {chosen['name']}  ({IMG_WIDTH}x{IMG_HEIGHT} @ {FPS} fps)")
print(f"Mode     : {PIPELINE_MODE}  (aspect {ASPECT_RATIO:.2f})")
print("Cell 2 complete.")


[basketball_side]  sport=basketball  view=side
  test/  (7 sequences)
      Q4_side_480-510                             frames: 900  gt:True  mp4:True
      Q4_side_510-540                             frames: 900  gt:True  mp4:True
      Q4_side_540-570                             frames: 900  gt:True  mp4:True
      Q4_side_570-600                             frames: 900  gt:True  mp4:True
      Q4_side_60-90                               frames: 901  gt:True  mp4:True
      Q4_side_600-end                             frames:  50  gt:True  mp4:True
      Q4_side_90-120                              frames: 901  gt:True  mp4:True
  train/  (35 sequences)
      Q1_side_0-30                                frames: 720  gt:True  mp4:True
      Q1_side_120-150                             frames: 721  gt:True  mp4:True
      Q1_side_150-180                             frames: 721  gt:True  mp4:True
      Q1_side_180-210                             frames: 720  gt:True  mp4:True
      Q1_side

In [31]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 3 — Model & Tracker Initialization  (FIX: meaningful conf split)
# ────────────────────────────────────────────────────────────────────────────────

# ── 3.1  YOLOv8m player detector ─────────────────────────────────────────────────────
MODEL = YOLO("yolov8m.pt")
MODEL.to(device)
PERSON_CLASS_ID = 0

if USE_SAHI:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    SAHI_MODEL = AutoDetectionModel.from_pretrained(
        model_type="ultralytics", model_path="yolov8m.pt",
        confidence_threshold=DET_CONF, device=device,
    )
else:
    SAHI_MODEL = None

def run_detection(frame):
    if USE_SAHI:
        res = get_sliced_prediction(
            frame, SAHI_MODEL,
            slice_height=SAHI_SLICE_H, slice_width=SAHI_SLICE_W,
            overlap_height_ratio=SAHI_OVERLAP, overlap_width_ratio=SAHI_OVERLAP,
            verbose=0,
        )
        boxes, confs, cids = [], [], []
        for o in res.object_prediction_list:
            if o.score.value < DET_CONF or o.category.name.lower() != "person":
                continue
            b = o.bbox
            boxes.append([b.minx, b.miny, b.maxx, b.maxy])
            confs.append(o.score.value)
            cids.append(o.category.id)
        if not boxes:
            return sv.Detections.empty()
        return sv.Detections(
            xyxy=np.array(boxes, dtype=np.float32),
            confidence=np.array(confs, dtype=np.float32),
            class_id=np.array(cids, dtype=int),
        )
    else:
        r = MODEL(frame, conf=DET_CONF, iou=DET_IOU,
                  classes=[PERSON_CLASS_ID], verbose=False)[0]
        return sv.Detections.from_ultralytics(r)

# ── 3.2  ByteTrack — FIX: wider conf split so two-pass actually rescues ────────
#   track_activation_threshold = 0.35  → "high-conf" pool
#   second-pass threshold is internal (0.15) → "low-conf" rescue pool
#   This creates a meaningful split instead of both pools being identical.
BT_HIGH_CONF = 0.35 if IS_PANORAMIC else 0.45    # first-pass threshold
BT_LOW_CONF  = DET_CONF                           # second-pass (rescue) threshold

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    TRACKER = sv.ByteTrack(
        track_activation_threshold=BT_HIGH_CONF,   # ← was DET_CONF (0.15) — fixed
        lost_track_buffer=30,
        minimum_matching_threshold=0.8,
        frame_rate=FPS,
    )

print(f"ByteTrack: high-conf threshold = {BT_HIGH_CONF}  |  rescue threshold = {BT_LOW_CONF}")
print("  → Two pools are now meaningfully different; rescue pass will fire.")

BOX_ANNOTATOR   = sv.BoxAnnotator(thickness=2)
LABEL_ANNOTATOR = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
TRACE_ANNOTATOR = sv.TraceAnnotator(thickness=2, trace_length=30)

# ── 3.3  Smoke test ──────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_FILE))
TRACKER.reset()
for target_idx in [0, 50, 100]:
    TRACKER.reset()
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
    ret, frame = cap.read()
    if not ret:
        continue
    dets = run_detection(frame)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        tracked = TRACKER.update_with_detections(dets)
    top5 = sorted(dets.confidence, reverse=True)[:5] if len(dets) > 0 else []
    print(f"Frame {target_idx:>4}: {len(dets):>3} dets → {len(tracked):>3} tracks  top5={[round(float(c),3) for c in top5]}")
cap.release()
TRACKER.reset()
print("Cell 3 complete.")

ByteTrack: high-conf threshold = 0.35  |  rescue threshold = 0.15
  → Two pools are now meaningfully different; rescue pass will fire.
Frame    0:  33 dets →  22 tracks  top5=[0.872, 0.85, 0.837, 0.83, 0.829]
Frame   50:  35 dets →  24 tracks  top5=[0.866, 0.844, 0.818, 0.817, 0.809]
Frame  100:  36 dets →  26 tracks  top5=[0.888, 0.858, 0.854, 0.838, 0.832]
Cell 3 complete.


In [32]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 4 — Detection, Tracking & CSV  (unchanged logic; uses new TRACKER)
# ────────────────────────────────────────────────────────────────────────────────
TRACKED_VIDEO_PATH = VIDEO_OUT_DIR / f"{SELECTED_SEQ.name}_tracked.mp4"
TRACKING_CSV_PATH  = STATS_OUT_DIR / f"{SELECTED_SEQ.name}_tracks.csv"

cap          = cv2.VideoCapture(str(VIDEO_FILE))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
SRC_FPS      = cap.get(cv2.CAP_PROP_FPS) or FPS
SRC_W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
SRC_H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
MAX_FRAMES   = 700 if IS_PANORAMIC else None
FRAMES_TO_RUN = min(MAX_FRAMES, TOTAL_FRAMES) if MAX_FRAMES else TOTAL_FRAMES

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(TRACKED_VIDEO_PATH), fourcc, SRC_FPS, (SRC_W, SRC_H))

TRACKER.reset()
tracking_rows = []
frame_idx = 0
t_start   = time.time()

try:
    while frame_idx < FRAMES_TO_RUN:
        ret, frame = cap.read()
        if not ret:
            break
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            dets = run_detection(frame)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            tracked = TRACKER.update_with_detections(dets)
        for i in range(len(tracked)):
            x1, y1, x2, y2 = tracked.xyxy[i]
            tracking_rows.append({
                "frame": frame_idx, "track_id": int(tracked.tracker_id[i]),
                "x1": round(float(x1),1), "y1": round(float(y1),1),
                "x2": round(float(x2),1), "y2": round(float(y2),1),
                "cx": round(float((x1+x2)/2),1), "cy": round(float((y1+y2)/2),1),
                "conf": round(float(tracked.confidence[i]),3),
                "width": round(float(x2-x1),1), "height": round(float(y2-y1),1),
            })
        labels    = [f"#{int(tid)}" for tid in tracked.tracker_id]
        annotated = frame.copy()
        annotated = TRACE_ANNOTATOR.annotate(annotated, tracked)
        annotated = BOX_ANNOTATOR.annotate(annotated, tracked)
        annotated = LABEL_ANNOTATOR.annotate(annotated, tracked, labels=labels)
        cv2.putText(annotated, f"[{PIPELINE_MODE}] Frame {frame_idx:04d} | Tracks:{len(tracked)}",
                    (12,32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2, cv2.LINE_AA)
        writer.write(annotated)
        if frame_idx % 25 == 0:
            elapsed = time.time() - t_start
            print(f"Frame {frame_idx:>5}  dets:{len(dets):>3}  tracks:{len(tracked):>3}  {elapsed:.0f}s")
        frame_idx += 1
finally:
    cap.release()
    writer.release()

TRACKS_DF = pd.DataFrame(tracking_rows)
TRACKS_DF.to_csv(TRACKING_CSV_PATH, index=False)
print(f"\nTracking CSV: {TRACKING_CSV_PATH}  ({len(TRACKS_DF)} rows, "
      f"{TRACKS_DF['track_id'].nunique()} unique IDs)")
print("Cell 4 complete.")

Frame     0  dets: 33  tracks: 22  2s
Frame    25  dets: 35  tracks: 24  47s
Frame    50  dets: 35  tracks: 26  91s
Frame    75  dets: 34  tracks: 26  136s
Frame   100  dets: 36  tracks: 26  181s
Frame   125  dets: 33  tracks: 26  226s
Frame   150  dets: 34  tracks: 27  271s
Frame   175  dets: 37  tracks: 28  315s
Frame   200  dets: 39  tracks: 28  361s
Frame   225  dets: 34  tracks: 28  406s
Frame   250  dets: 37  tracks: 29  451s
Frame   275  dets: 38  tracks: 29  496s
Frame   300  dets: 39  tracks: 30  541s
Frame   325  dets: 38  tracks: 29  585s
Frame   350  dets: 35  tracks: 28  630s
Frame   375  dets: 31  tracks: 25  675s
Frame   400  dets: 32  tracks: 25  720s
Frame   425  dets: 35  tracks: 24  765s
Frame   450  dets: 35  tracks: 26  809s
Frame   475  dets: 35  tracks: 27  854s
Frame   500  dets: 37  tracks: 27  899s
Frame   525  dets: 33  tracks: 26  943s
Frame   550  dets: 33  tracks: 29  989s
Frame   575  dets: 35  tracks: 29  1033s
Frame   600  dets: 37  tracks: 28  1078s
Fr

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 5 — Ball Detection  (FIX: fine-tuned model + Kalman gate filter)
# ────────────────────────────────────────────────────────────────────────────────

# ── 5.1  Download the Roboflow/SkalskiP fine-tuned ball detector ─────────────
#   This model is trained specifically on wide-angle sports footage where
#   the ball is 8-15px wide — COCO class 32 on yolov8x is NOT good enough
#   for panoramic views and causes the "ghost dot" problem.

import urllib.request

BALL_MODEL_PATH = Path("./ball_detector.pt")

if not BALL_MODEL_PATH.exists():
    print("Downloading fine-tuned football ball detector ...")
    try:
        urllib.request.urlretrieve(
            "https://huggingface.co/SkalskiP/sports.cv/resolve/main/football-ball-detection.pt",
            str(BALL_MODEL_PATH)
        )
        print(f"  Downloaded → {BALL_MODEL_PATH}")
    except Exception as e:
        print(f"  Download failed ({e}) — falling back to yolov8x COCO class 32")
        BALL_MODEL_PATH = None

if BALL_MODEL_PATH and BALL_MODEL_PATH.exists():
    BALL_MODEL    = YOLO(str(BALL_MODEL_PATH))
    BALL_CLASS_ID = 0      # fine-tuned model has single class: ball
    BALL_CONF     = 0.25
    print("Ball model: Roboflow fine-tuned (wide-angle)")
else:
    BALL_MODEL    = YOLO("yolov8x.pt")
    BALL_CLASS_ID = 32     # COCO sports ball
    BALL_CONF     = 0.10
    print("Ball model: YOLOv8x COCO fallback (expect ghost-dot artefacts)")

BALL_MODEL.to(device)

# ── 5.2  Kalman filter for ball trajectory smoothing + ghost-dot rejection ────
#   State:  [x, y, vx, vy]
#   Measurement: [x, y]
#   Rejects any detection that jumps more than MAX_BALL_JUMP pixels from
#   the Kalman-predicted position — this eliminates the ghost dot while
#   keeping the true trajectory intact.

ball_kf = KalmanFilter(dim_x=4, dim_z=2)
ball_kf.F = np.array([[1,0,1,0],
                       [0,1,0,1],
                       [0,0,1,0],
                       [0,0,0,1]], dtype=float)
ball_kf.H = np.array([[1,0,0,0],
                       [0,1,0,0]], dtype=float)
ball_kf.R *= 50
ball_kf.P *= 500

MAX_BALL_JUMP   = 120   # px — reject if ball teleports more than this per frame
ball_kf_init    = False # track whether KF has been seeded

def update_ball_kf(cx, cy):
    """
    Feed a candidate ball detection through the Kalman gate.
    Returns True (accepted, KF updated) or False (ghost dot, rejected).
    """
    global ball_kf_init
    ball_kf.predict()
    if not ball_kf_init:
        # Seed the filter with the first detection
        ball_kf.x[:2] = np.array([[cx], [cy]])
        ball_kf_init  = True
        ball_kf.update(np.array([[cx],[cy]]))
        return True
    pred = ball_kf.x[:2].flatten()
    dist = np.linalg.norm(np.array([cx, cy]) - pred)
    if dist < MAX_BALL_JUMP:
        ball_kf.update(np.array([[cx],[cy]]))
        return True
    return False  # ghost dot rejected

# ── 5.3  SAHI wrapper for ball (fine-tuned model is small; SAHI still helps) ─
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

BALL_SAHI_MODEL = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(BALL_MODEL_PATH) if BALL_MODEL_PATH and BALL_MODEL_PATH.exists() else "yolov8x.pt",
    confidence_threshold=BALL_CONF,
    device=device,
)
BALL_SAHI_SLICE   = 320
BALL_SAHI_OVERLAP = 0.2

def detect_ball(frame):
    """
    Detect ball in a single frame. Returns (cx, cy, conf) or None.
    Ghost dots are rejected by the Kalman gate.
    """
    result = get_sliced_prediction(
        frame, BALL_SAHI_MODEL,
        slice_height=BALL_SAHI_SLICE, slice_width=BALL_SAHI_SLICE,
        overlap_height_ratio=BALL_SAHI_OVERLAP, overlap_width_ratio=BALL_SAHI_OVERLAP,
        verbose=0,
    )
    ball_preds = [
        o for o in result.object_prediction_list
        if o.category.id == BALL_CLASS_ID and o.score.value >= BALL_CONF
    ]
    if not ball_preds:
        return None
    best = max(ball_preds, key=lambda o: o.score.value)
    b    = best.bbox
    cx   = (b.minx + b.maxx) / 2
    cy   = (b.miny + b.maxy) / 2
    if not update_ball_kf(cx, cy):
        return None   # Kalman gate rejected — ghost dot
    return (round(cx,1), round(cy,1), round(best.score.value,3))

# ── 5.4  Full video ball detection loop ─────────────────────────────────────────────────────────────
BALL_CSV_PATH = STATS_OUT_DIR / f"{SELECTED_SEQ.name}_ball.csv"
cap  = cv2.VideoCapture(str(VIDEO_FILE))
ball_rows  = []
frame_idx  = 0
detected   = 0
t_start    = time.time()

while frame_idx < FRAMES_TO_RUN:
    ret, frame = cap.read()
    if not ret:
        break
    result = detect_ball(frame)
    if result:
        cx, cy, conf = result
        detected += 1
        ball_rows.append({"frame": frame_idx, "cx": cx, "cy": cy, "conf": conf})
    else:
        ball_rows.append({"frame": frame_idx, "cx": float("nan"),
                          "cy": float("nan"), "conf": float("nan")})
    if frame_idx % 50 == 0:
        elapsed = time.time() - t_start
        print(f"  Frame {frame_idx:>4} | detected:{detected} | {elapsed:.0f}s")
    frame_idx += 1

cap.release()
BALL_DF = pd.DataFrame(ball_rows)
BALL_DF.to_csv(BALL_CSV_PATH, index=False)
print(f"\nBall: {detected}/{frame_idx} frames detected ({detected/frame_idx*100:.1f}%)")
print("Cell 5 complete.")

  Download failed (HTTP Error 401: Unauthorized) — falling back to yolov8x COCO class 32
Ball model: YOLOv8x COCO fallback (expect ghost-dot artefacts)
  Frame    0 | detected:0 | 8s
  Frame   50 | detected:16 | 396s
  Frame  100 | detected:46 | 784s
  Frame  150 | detected:96 | 1171s
  Frame  200 | detected:146 | 1558s
  Frame  250 | detected:196 | 1946s
  Frame  300 | detected:246 | 2332s


In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 6 — Team Clustering  (FIX: per-ID centroids from non-overlapping frames)
# ────────────────────────────────────────────────────────────────────────────────
# Problem: per-frame KMeans fails on overlapping crops.
# Fix: build per-track color histograms only from frames where that player
#      does NOT overlap with any other player, then classify via
#      nearest-centroid (not per-frame KMeans).

N_TEAMS         = 2
CROP_TORSO_FRAC = 0.4
MIN_CROP_PX     = 10
OVERLAP_THRESH  = 0.3   # IoU threshold to flag two players as overlapping

def _iou(b1, b2):
    xx1 = max(b1[0], b2[0]); yy1 = max(b1[1], b2[1])
    xx2 = min(b1[2], b2[2]); yy2 = min(b1[3], b2[3])
    w = max(0., xx2-xx1); h = max(0., yy2-yy1)
    inter = w * h
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    denom = a1 + a2 - inter
    return inter / denom if denom > 0 else 0.

def extract_upper_body_crop(frame, row):
    x1,y1,x2,y2 = int(row["x1"]),int(row["y1"]),int(row["x2"]),int(row["y2"])
    h_f,w_f = frame.shape[:2]
    x1 = max(0,x1); y1 = max(0,y1); x2 = min(w_f,x2); y2 = min(h_f,y2)
    bh = y2-y1; bw = x2-x1
    if bh < MIN_CROP_PX or bw < MIN_CROP_PX:
        return None
    ty1 = y1 + int(bh*0.25); ty2 = y1 + int(bh*0.65)
    crop = frame[ty1:ty2, x1:x2]
    return crop if crop.size > 0 else None

def build_per_id_color_feature(frame, row):
    """RGB mean + HSV mean on torso — 6-dim feature vector."""
    crop = extract_upper_body_crop(frame, row)
    if crop is None:
        return None
    rgb_mean = crop.mean(axis=(0,1))
    hsv      = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    return np.array([
        rgb_mean[2], rgb_mean[1], rgb_mean[0],
        hsv[:,:,0].mean(), hsv[:,:,1].mean(), hsv[:,:,2].mean()
    ], dtype=np.float32)

# ── 6.1  Accumulate per-ID features (skip overlapping frames) ────────────────
print("Building per-ID color features from non-overlapping frames ...")
per_id_feats = defaultdict(list)

cap = cv2.VideoCapture(str(VIDEO_FILE))
prev_fidx = -1
cur_frame = None

for fidx, group in sorted(TRACKS_DF.groupby("frame"), key=lambda x: x[0]):
    if fidx != prev_fidx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
        ret, cur_frame = cap.read()
        if not ret:
            continue
        prev_fidx = fidx

    boxes = group[["x1","y1","x2","y2"]].values
    ids   = group["track_id"].values
    # Flag overlapping pairs
    overlapping = set()
    for i in range(len(boxes)):
        for j in range(i+1, len(boxes)):
            if _iou(boxes[i], boxes[j]) > OVERLAP_THRESH:
                overlapping.add(ids[i]); overlapping.add(ids[j])

    for _, row in group.iterrows():
        if row["track_id"] in overlapping:
            continue   # skip — jersey colour unreliable when overlapping
        feat = build_per_id_color_feature(cur_frame, row)
        if feat is not None:
            per_id_feats[int(row["track_id"])].append(feat)

cap.release()

# ── 6.2  Average features per ID, then cluster ──────────────────────────────────
valid_ids  = []
features   = []
for tid, feats in per_id_feats.items():
    if len(feats) > 0:
        valid_ids.append(tid)
        features.append(np.mean(feats, axis=0))

if len(features) < N_TEAMS:
    raise RuntimeError(f"Only {len(features)} IDs with clean crops — lower OVERLAP_THRESH?")

X = np.array(features)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

best_inertia = float("inf")
best_labels  = None
for seed in range(10):
    km = KMeans(n_clusters=N_TEAMS, random_state=seed, n_init=20)
    lbl = km.fit_predict(X_scaled)
    if km.inertia_ < best_inertia:
        best_inertia = km.inertia_; best_labels = lbl

counts = Counter(best_labels)
# Ensure Team 0 = larger cluster
if counts[0] < counts[1]:
    best_labels = 1 - best_labels

team_map = {valid_ids[i]: int(best_labels[i]) for i in range(len(valid_ids))}
# Assign IDs with no clean crops to team -1
for tid in TRACKS_DF["track_id"].unique():
    if tid not in team_map:
        team_map[int(tid)] = -1

TRACKS_DF["team"] = TRACKS_DF["track_id"].map(team_map).fillna(-1).astype(int)
print(f"Team A: {(TRACKS_DF['team']==0).sum()} rows | Team B: {(TRACKS_DF['team']==1).sum()} rows "
      f"| Unknown: {(TRACKS_DF['team']==-1).sum()} rows")

# ── 6.3  Outlier detection (referee / GK) ───────────────────────────────────────
outlier_ids = set()
for team_id in [0,1]:
    tidxs = [i for i,tid in enumerate(valid_ids) if team_map.get(tid,-1)==team_id]
    if not tidxs:
        continue
    tf = X_scaled[tidxs]
    centroid  = tf.mean(axis=0, keepdims=True)
    distances = cdist(tf, centroid, metric="euclidean").flatten()
    thresh    = distances.mean() + 1.5 * distances.std()
    for idx, dist in zip(tidxs, distances):
        if dist > thresh:
            outlier_ids.add(valid_ids[idx])

TRACKS_DF["is_outlier"] = TRACKS_DF["track_id"].isin(outlier_ids)
print(f"Outliers flagged: {sorted(outlier_ids)}")

TEAM_CSV = STATS_OUT_DIR / f"{SELECTED_SEQ.name}_teams.csv"
TRACKS_DF.to_csv(TEAM_CSV, index=False)
print("Cell 6 complete.")

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 6B — Re-ID Feature Extraction  (NEW — for overlap survival)
# ────────────────────────────────────────────────────────────────────────────────
# Uses OSNet-AIN to build a 512-dim embedding per track ID from the best
# (highest-confidence, non-overlapping) crop of each player.
# These embeddings are then used in Cell 10 to augment the SORT cost matrix,
# so track IDs survive full occlusion.

try:
    import torchreid
    REID_AVAILABLE = True
except ImportError:
    REID_AVAILABLE = False
    print("torchreid not installed — pip install torchreid to enable Re-ID.\n"
          "Falling back to IoU-only matching for overlap cases.")

if REID_AVAILABLE:
    reid_model = torchreid.models.build_model(
        name="osnet_ain_x1_0", num_classes=1000, pretrained=True
    )
    reid_model.eval()
    if device == "cuda":
        reid_model = reid_model.cuda()

    import torchvision.transforms as T
    reid_transform = T.Compose([
        T.ToPILImage(),
        T.Resize((256, 128)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    def get_reid_feature(frame, row):
        """
        Extract a 512-dim embedding for one player crop.
        Returns None if crop is too small.
        """
        x1,y1,x2,y2 = int(row["x1"]),int(row["y1"]),int(row["x2"]),int(row["y2"])
        h_f,w_f = frame.shape[:2]
        x1 = max(0,x1); y1 = max(0,y1); x2 = min(w_f,x2); y2 = min(h_f,y2)
        if (y2-y1) < 20 or (x2-x1) < 10:
            return None
        crop = frame[y1:y2, x1:x2]
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        tensor = reid_transform(crop_rgb).unsqueeze(0)
        if device == "cuda":
            tensor = tensor.cuda()
        with torch.no_grad():
            feat = reid_model(tensor)
        return feat.cpu().numpy().flatten()

    # Build per-ID Re-ID embedding from best clean crop
    print("Extracting Re-ID embeddings (OSNet-AIN) ...")
    best_frames = (
        TRACKS_DF[~TRACKS_DF["is_outlier"]]
        .sort_values("conf", ascending=False)
        .groupby("track_id").first().reset_index()
    )
    reid_map = {}   # track_id → 512-dim feature
    cap = cv2.VideoCapture(str(VIDEO_FILE))
    prev_fidx = -1; cur_frame = None
    for _, row in best_frames.iterrows():
        fidx = int(row["frame"])
        if fidx != prev_fidx:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
            ret, cur_frame = cap.read()
            if not ret: continue
            prev_fidx = fidx
        feat = get_reid_feature(cur_frame, row)
        if feat is not None:
            reid_map[int(row["track_id"])] = feat
    cap.release()
    print(f"  Re-ID embeddings built for {len(reid_map)} track IDs.")
else:
    reid_map = {}

print("Cell 6B complete.")

In [8]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 7 — Ball Possession %  (unchanged — uses cleaned TRACKS_DF)
# ────────────────────────────────────────────────────────────────────────────────
MAX_POSSESSION_DIST = 150
ball_detected = BALL_DF[BALL_DF["conf"].notna()].copy()
possession_rows = []

for _, ball_row in ball_detected.iterrows():
    fid     = int(ball_row["frame"])
    bx, by  = ball_row["cx"], ball_row["cy"]
    players = TRACKS_DF[(TRACKS_DF["frame"]==fid) & (~TRACKS_DF["is_outlier"])]
    if players.empty:
        possession_rows.append({"frame":fid,"ball_cx":bx,"ball_cy":by,
                                 "closest_id":-1,"closest_dist":float("nan"),
                                 "team":-1,"possession":"no_data"}); continue
    dx = players["cx"] - bx; dy = players["cy"] - by
    dist = np.sqrt(dx**2 + dy**2)
    idx  = dist.idxmin(); md = dist[idx]; closest = players.loc[idx]
    if md <= MAX_POSSESSION_DIST:
        team = int(closest["team"]); poss = {0:"Team A",1:"Team B"}.get(team,"unknown")
    else:
        team = -1; poss = "contested"
    possession_rows.append({"frame":fid,"ball_cx":bx,"ball_cy":by,
                             "closest_id":int(closest["track_id"]),"closest_dist":round(float(md),1),
                             "team":team,"possession":poss})

POSSESSION_DF = pd.DataFrame(possession_rows)
counts   = POSSESSION_DF["possession"].value_counts()
a_f, b_f = counts.get("Team A",0), counts.get("Team B",0)
assigned = a_f + b_f
if assigned > 0:
    print(f"Possession — Team A: {a_f/assigned*100:.1f}%  Team B: {b_f/assigned*100:.1f}%")
POSSESSION_CSV = STATS_OUT_DIR / f"{SELECTED_SEQ.name}_possession.csv"
POSSESSION_DF.to_csv(POSSESSION_CSV, index=False)
print("Cell 7 complete.")

Possession — Team A: 69.1%  Team B: 30.9%
Cell 7 complete.


In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 8 — Heatmap + Ball Trajectory  (unchanged except trajectory uses KF)
# ────────────────────────────────────────────────────────────────────────────────
from scipy.stats import gaussian_kde
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle

HEATMAP_RESOLUTION = (650, 100)
KDE_BANDWIDTH      = 0.12
tracks_clean = TRACKS_DF[~TRACKS_DF["is_outlier"]].copy()
team_a = tracks_clean[tracks_clean["team"]==0]
team_b = tracks_clean[tracks_clean["team"]==1]

def draw_pitch(ax, pw=6500, ph=1000):
    ax.set_facecolor("#2d8a4e")
    lw = 2.5; lc = "white"
    ax.plot([0,pw,pw,0,0],[0,0,ph,ph,0],color=lc,lw=lw)
    ax.plot([pw/2,pw/2],[0,ph],color=lc,lw=lw)
    ax.add_patch(Circle((pw/2,ph/2),radius=ph*0.18,fill=False,color=lc,lw=lw))
    ax.plot(pw/2,ph/2,"o",color=lc,markersize=4)
    for x0,sign in [(0,1),(pw,-1)]:
        pb_w=pw*0.12; pb_h=ph*0.6; pb_y=(ph-pb_h)/2
        ax.plot([x0,x0+sign*pb_w,x0+sign*pb_w,x0],[pb_y,pb_y,pb_y+pb_h,pb_y+pb_h],color=lc,lw=lw)
        gb_w=pw*0.04; gb_h=ph*0.28; gb_y=(ph-gb_h)/2
        ax.plot([x0,x0+sign*gb_w,x0+sign*gb_w,x0],[gb_y,gb_y,gb_y+gb_h,gb_y+gb_h],color=lc,lw=lw)
    ax.set_xlim(0,pw); ax.set_ylim(0,ph); ax.set_aspect("equal"); ax.axis("off")

def compute_kde(xs, ys, gw, gh, bw):
    if len(xs) < 5:
        return np.zeros((gh,gw))
    xn = np.array(xs)/IMG_WIDTH; yn = np.array(ys)/IMG_HEIGHT
    kde = gaussian_kde(np.vstack([xn,yn]), bw_method=bw)
    xi = np.linspace(0,1,gw); yi = np.linspace(0,1,gh)
    Xi,Yi = np.meshgrid(xi,yi)
    return kde(np.vstack([Xi.ravel(),Yi.ravel()])).reshape(gh,gw)

cmap_a = LinearSegmentedColormap.from_list("ta",["#000080","#0000FF","#00FFFF","#FFFF00"],N=256)
cmap_b = LinearSegmentedColormap.from_list("tb",["#800000","#FF0000","#FF8000","#FFFF00"],N=256)

Z_a = compute_kde(team_a["cx"],team_a["cy"],*HEATMAP_RESOLUTION,KDE_BANDWIDTH)
Z_b = compute_kde(team_b["cx"],team_b["cy"],*HEATMAP_RESOLUTION,KDE_BANDWIDTH)

fig, axes = plt.subplots(1,2,figsize=(20,4))
for ax,Z,cmap,title in [(axes[0],Z_a,cmap_a,"Team A"),(axes[1],Z_b,cmap_b,"Team B")]:
    draw_pitch(ax)
    Zp = Z/Z.max() if Z.max()>0 else Z
    ax.imshow(Zp,extent=[0,IMG_WIDTH,IMG_HEIGHT,0],origin="upper",cmap=cmap,
              alpha=0.65,aspect="auto",interpolation="bilinear")
    ax.set_title(title,fontsize=12)
plt.tight_layout()
plt.savefig(HEAT_OUT_DIR/f"{SELECTED_SEQ.name}_heatmap_teams.png",dpi=150,bbox_inches="tight")
plt.show()

# Ball trajectory — Kalman-smoothed (ghost dots already removed in Cell 5)
ball_valid  = BALL_DF[BALL_DF["conf"].notna()].sort_values("frame")
all_frames  = pd.DataFrame({"frame": range(FRAMES_TO_RUN)})
ball_interp = all_frames.merge(ball_valid,on="frame",how="left")
ball_interp["cx"] = ball_interp["cx"].interpolate(method="linear")
ball_interp["cy"] = ball_interp["cy"].interpolate(method="linear")

from matplotlib.collections import LineCollection
fig, ax = plt.subplots(figsize=(20,4))
draw_pitch(ax)
pts = np.array([ball_interp["cx"],ball_interp["cy"]]).T.reshape(-1,1,2)
segs = np.concatenate([pts[:-1],pts[1:]],axis=1)
lc2  = LineCollection(segs,cmap=plt.cm.plasma,norm=plt.Normalize(0,len(ball_interp)),linewidth=2,alpha=0.85)
lc2.set_array(np.arange(len(ball_interp)))
ax.add_collection(lc2)
ax.plot(*ball_interp.iloc[0][["cx","cy"]],"o",color="white",markersize=8,zorder=5)
ax.plot(*ball_interp.iloc[-1][["cx","cy"]],"s",color="yellow",markersize=8,zorder=5)
ax.set_title("Ball Trajectory — Kalman-smoothed (ghost dots removed)",fontsize=12,fontweight="bold")
plt.tight_layout()
plt.savefig(HEAT_OUT_DIR/f"{SELECTED_SEQ.name}_ball_trajectory.png",dpi=150,bbox_inches="tight")
plt.show()
print("Cell 8 complete.")

In [ ]:
# =============================================================================
# CELL 8B — Automatic Homography via Pitch Contour (panoramic, Kaggle‑safe)
# =============================================================================
import cv2, numpy as np, matplotlib.pyplot as plt

HOMOG_OK = False
H = None
H_inv = None

# 1. Read first frame
cap = cv2.VideoCapture(str(VIDEO_FILE))
ret, frame = cap.read()
cap.release()
if not ret:
    raise RuntimeError("Could not read video frame.")

h, w = frame.shape[:2]
rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# 2. Segment green pitch – robust to different lighting
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
green_lower = np.array([25, 30, 40])
green_upper = np.array([95, 255, 255])
green_mask = cv2.inRange(hsv, green_lower, green_upper)

# Clean up mask
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_CLOSE, kernel)
green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_OPEN, kernel)

# 3. Find all contours on the green mask
contours, _ = cv2.findContours(green_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contours:
    print("❌ No green region found. Falling back to manual calibration.")
else:
    # Take the largest contour (the field)
    largest = max(contours, key=cv2.contourArea)

    # 4. Approximate as a 4‑point polygon (should be the field outline)
    peri = cv2.arcLength(largest, True)
    # We want exactly 4 corners → adjust epsilon until we get 4
    for eps_factor in [0.02, 0.05, 0.1]:
        approx = cv2.approxPolyDP(largest, eps_factor * peri, True)
        if len(approx) == 4:
            break

    # If still not 4, take the convex hull and force 4 points (fallback)
    if len(approx) != 4:
        hull = cv2.convexHull(largest)
        approx = cv2.approxPolyDP(hull, 0.02 * cv2.arcLength(hull, True), True)
        if len(approx) != 4:
            # Force to 4 by extreme points of hull
            rect = cv2.minAreaRect(hull)
            approx = cv2.boxPoints(rect).astype(np.int32)
            approx = approx.reshape(4,1,2)

    # 5. Order points consistently: top‑left, top‑right, bottom‑right, bottom‑left
    pts = approx[:,0,:].astype(np.float32)   # (4,2)
    # Sort by y (top to bottom)
    pts = pts[np.argsort(pts[:,1])]
    top_two = pts[:2][np.argsort(pts[:2,0])]      # smaller x first = top‑left, then top‑right
    bottom_two = pts[2:][np.argsort(pts[2:,0])[::-1]]  # larger x first = bottom‑right, then bottom‑left
    ordered = np.float32([top_two[0], top_two[1], bottom_two[0], bottom_two[1]])

    src_pts = ordered.reshape(-1, 1, 2)
    # Real pitch corners – FIFA standard 105m x 68m (in the order above)
    dst_pts = np.float32([
        [  0.0,  0.0],    # top‑left
        [105.0,  0.0],    # top‑right
        [105.0, 68.0],    # bottom‑right
        [  0.0, 68.0]     # bottom‑left
    ]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    if H is not None:
        H_inv = np.linalg.inv(H)
        HOMOG_OK = True
        print("✅ Homography successfully computed from field outline!")
    else:
        print("❌ Homography could not be computed despite finding field contour.")

# 6. Visual check – always show the detected outline
fig, ax = plt.subplots(figsize=(16, 6))
ax.imshow(rgb)
if HOMOG_OK:
    # Draw the detected corners
    ax.scatter(src_pts[:,0,0], src_pts[:,0,1], c='cyan', s=80)
    # Project the real pitch outline
    pitch_outline = np.float32([[0,0],[105,0],[105,68],[0,68]]).reshape(-1,1,2)
    proj = cv2.perspectiveTransform(pitch_outline, H_inv).astype(int)
    cv2.polylines(frame, [proj], True, (0,255,0), 3)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title("Auto‑detected pitch (green outline) and corners (cyan dots)")
else:
    # If failed, just show the mask outline to debug
    rgb_copy = rgb.copy()
    cv2.drawContours(rgb_copy, [approx], -1, (0,0,255), 3)
    ax.imshow(rgb_copy)
    ax.set_title("Detected green contour (red) – check if it matches the pitch")
plt.show()

# Convenience wrappers (same as always)
def img_to_pitch(px, py):
    if not HOMOG_OK:
        raise RuntimeError("Homography not calibrated – see Cell 8B.")
    pt = np.array([[[float(px), float(py)]]], dtype=np.float32)
    return cv2.perspectiveTransform(pt, H)[0][0]

def pitch_to_img(rx, ry):
    if not HOMOG_OK:
        raise RuntimeError("Homography not calibrated – see Cell 8B.")
    pt = np.array([[[float(rx), float(ry)]]], dtype=np.float32)
    return cv2.perspectiveTransform(pt, H_inv)[0][0].astype(int)

print("Cell 8B complete – HOMOG_OK =", HOMOG_OK)

In [ ]:
# =============================================================================
# CELL 9 — Final Annotated Video  (offside line removed; triangle kept)
# =============================================================================
import imageio

FINAL_VIDEO_PATH = VIDEO_OUT_DIR / f"{SELECTED_SEQ.name}_final.mp4"

TEAM_COLOR_BGR = {0:(255,80,20), 1:(20,20,220), -1:(128,128,128)}
TEAM_NAME      = {0:"Team A", 1:"Team B", -1:"?"}

frame_to_tracks = {}
for _, row in TRACKS_DF.iterrows():
    frame_to_tracks.setdefault(int(row["frame"]),[]).append(row)

frame_to_ball = {}
for _, row in BALL_DF.iterrows():
    if not np.isnan(row["cx"]):
        frame_to_ball[int(row["frame"])] = (int(row["cx"]), int(row["cy"]))

frame_to_poss = {int(r["frame"]): r["possession"] for _, r in POSSESSION_DF.iterrows()}

# ── 9.2  Defensive triangle overlay ───────────────────────────────────────────
def draw_defensive_triangle(frame, tracks_list, attacking_team_id):
    """
    Draw the triangular shape connecting the last 3 defenders (closest to own goal).
    Requires homography calibration.
    """
    if not HOMOG_OK:
        return frame
    defenders = [r for r in tracks_list
                 if int(r["team"]) == (1 - attacking_team_id) and not r["is_outlier"]]
    if len(defenders) < 3:
        return frame
    try:
        def pitch_y(r):
            _, ry = img_to_pitch(float(r["cx"]), float(r["y2"]))
            return ry
        defenders_sorted = sorted(defenders, key=pitch_y, reverse=True)
        last3 = defenders_sorted[:3]   # 3 deepest (closest to own goal)
        pts = np.array([[int(r["cx"]), int(r["y2"])] for r in last3], np.int32)
        # Semi-transparent orange fill
        overlay = frame.copy()
        cv2.fillPoly(overlay, [pts.reshape(-1,1,2)], (255,165,0))
        cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, frame)
        # Outline
        cv2.polylines(frame, [pts.reshape(-1,1,2)], True, (255,165,0), 2, cv2.LINE_AA)
    except Exception:
        pass
    return frame

# ── 9.3  Stats overlay bar ────────────────────────────────────────────────────
def draw_stats_overlay(frame, frame_idx, poss_list):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    bar_h = max(40, int(h*0.08))
    cv2.rectangle(overlay,(0,h-bar_h),(w,h),(20,20,20),-1)
    cv2.addWeighted(overlay,0.6,frame,0.4,0,frame)
    if not poss_list:
        return frame
    cnt = Counter(poss_list)
    a,b = cnt.get("Team A",0), cnt.get("Team B",0)
    total = a+b
    if total==0: return frame
    bx1,bx2 = int(w*0.25),int(w*0.75); by1,by2 = h-bar_h+8, h-8
    bmid = int(bx1+(bx2-bx1)*(a/total))
    cv2.rectangle(frame,(bx1,by1),(bx2,by2),(50,50,50),-1)
    if bmid>bx1: cv2.rectangle(frame,(bx1,by1),(bmid,by2),(255,80,20),-1)
    if bmid<bx2: cv2.rectangle(frame,(bmid,by1),(bx2,by2),(20,20,220),-1)
    cv2.rectangle(frame,(bx1,by1),(bx2,by2),(200,200,200),1)
    font=cv2.FONT_HERSHEY_SIMPLEX; fs=max(0.4,h/2500)
    cv2.putText(frame,f"A {a/total*100:.0f}%",(bx1-int(w*0.08),by2-2),font,fs,(255,180,80),1,cv2.LINE_AA)
    cv2.putText(frame,f"B {b/total*100:.0f}%",(bx2+int(w*0.01),by2-2),font,fs,(80,80,255),1,cv2.LINE_AA)
    cv2.putText(frame,f"{frame_idx/FPS:.1f}s",(12,h-8),font,fs*0.9,(180,180,180),1,cv2.LINE_AA)
    return frame

# ── 9.4  Rendering loop (offside line completely removed) ─────────────────────
ATTACKING_TEAM = 0
cap        = cv2.VideoCapture(str(VIDEO_FILE))
frames_buf = []
frame_idx  = 0
poss_list  = []
t_start    = time.time()

while frame_idx < FRAMES_TO_RUN:
    ret, frame = cap.read()
    if not ret:
        break
    annotated = frame.copy()

    # Player boxes
    for row in frame_to_tracks.get(frame_idx, []):
        tid   = int(row["track_id"]); team = int(row["team"])
        is_out= bool(row["is_outlier"])
        color = (128,128,128) if is_out else TEAM_COLOR_BGR.get(team,(128,128,128))
        x1,y1,x2,y2 = int(row["x1"]),int(row["y1"]),int(row["x2"]),int(row["y2"])
        cv2.rectangle(annotated,(x1,y1),(x2,y2),color,2)
        lbl = f"#{tid}" if is_out else f"#{tid} {TEAM_NAME[team]}"
        cv2.putText(annotated,lbl,(max(0,x1),max(12,y1-4)),
                    cv2.FONT_HERSHEY_SIMPLEX,max(0.3,frame.shape[0]/3000),color,1,cv2.LINE_AA)

    # Ball
    ball = frame_to_ball.get(frame_idx)
    if ball:
        cv2.circle(annotated,ball,14,(0,255,0),3)
        cv2.circle(annotated,ball,3,(0,255,0),-1)
        cv2.putText(annotated,"ball",(ball[0]+16,ball[1]-8),
                    cv2.FONT_HERSHEY_SIMPLEX,max(0.35,frame.shape[0]/3000),(0,255,0),1,cv2.LINE_AA)

    # Defensive triangle only (offside line removed)
    tracks_now = frame_to_tracks.get(frame_idx, [])
    annotated = draw_defensive_triangle(annotated, tracks_now, ATTACKING_TEAM)

    # Possession accumulation
    poss = frame_to_poss.get(frame_idx)
    if poss in ("Team A","Team B"):
        poss_list.append(poss)
    annotated = draw_stats_overlay(annotated, frame_idx, poss_list)

    frames_buf.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

    if frame_idx % 50 == 0:
        print(f"  Frame {frame_idx:>4} | {time.time()-t_start:.0f}s elapsed")
    frame_idx += 1

cap.release()
imageio.mimwrite(str(FINAL_VIDEO_PATH), frames_buf, fps=int(FPS), codec="libx264", quality=7)
print(f"Final video: {FINAL_VIDEO_PATH}")
print("Cell 9 complete (offside line removed).")

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 10 — SORT Baseline  (FIX: Re-ID cost alongside IoU for overlap cases)
# ────────────────────────────────────────────────────────────────────────────────

REID_ALPHA = 0.4   # weight for Re-ID cosine distance in combined cost
                   # cost = REID_ALPHA * cosine_dist + (1-REID_ALPHA) * (1-IoU)
                   # Set to 0.0 to use pure IoU (original SORT behaviour)

def cosine_dist(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return 1.0 - float(np.dot(a, b))

class KalmanBoxTracker:
    count = 0
    def __init__(self, bbox):
        self.kf      = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F    = np.eye(7, dtype=np.float32)
        for i in range(4): self.kf.F[i,i+3] = 1.
        self.kf.H        = np.eye(4,7,dtype=np.float32)
        self.kf.R        = np.eye(4,dtype=np.float32); self.kf.R[2:,2:] *= 10.
        self.kf.P        = np.eye(7,dtype=np.float32); self.kf.P[4:,4:] *= 1000.; self.kf.P *= 10.
        self.kf.Q        = np.eye(7,dtype=np.float32); self.kf.Q[-1,-1] *= 0.01; self.kf.Q[4:,4:] *= 0.01
        x1,y1,x2,y2 = [float(v) for v in bbox]
        cx=(x1+x2)/2; cy=(y1+y2)/2; w=x2-x1; h=y2-y1; s=w*h; r=w/(h+1e-6)
        self.kf.x = np.array([[cx],[cy],[s],[r],[0.],[0.],[0.]],dtype=np.float32)
        self.time_since_update = 0
        self.id = KalmanBoxTracker.count; KalmanBoxTracker.count += 1
        self.hit_streak = 0

    def update(self, bbox):
        x1,y1,x2,y2 = [float(v) for v in bbox]
        cx=(x1+x2)/2; cy=(y1+y2)/2; w=x2-x1; h=y2-y1; s=w*h; r=w/(h+1e-6)
        self.kf.update(np.array([[cx],[cy],[s],[r]],dtype=np.float32))
        self.time_since_update=0; self.hit_streak+=1

    def predict(self):
        self.kf.predict(); self.time_since_update+=1; return self._state()

    def _state(self):
        x=self.kf.x.flatten(); cx,cy,s,r = float(x[0]),float(x[1]),float(x[2]),float(x[3])
        w=float(np.sqrt(max(s*r,1e-6))); h=float(np.sqrt(max(s/max(r,1e-6),1e-6)))
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]

def _iou_bb(a,b):
    xx1=max(a[0],b[0]);yy1=max(a[1],b[1]);xx2=min(a[2],b[2]);yy2=min(a[3],b[3])
    w=max(0.,xx2-xx1);h=max(0.,yy2-yy1);inter=w*h
    denom=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/denom if denom>0 else 0.

class SORTWithReID:
    """
    SORT tracker augmented with optional Re-ID embedding.
    When reid_map is provided and REID_ALPHA > 0, the cost matrix
    combines IoU distance with cosine Re-ID distance.
    This dramatically reduces ID switches during player overlap.
    """
    def __init__(self, max_age=3, min_hits=2, iou_thresh=0.25, reid_map=None, alpha=0.0):
        self.max_age=max_age; self.min_hits=min_hits; self.iou_thresh=iou_thresh
        self.trackers=[]; self.frame_count=0
        self.reid_map = reid_map or {}
        self.alpha    = alpha
        self.id_to_reid = {}   # internal id → last known Re-ID feature
        KalmanBoxTracker.count = 0

    def update(self, dets, det_ids=None):
        """
        dets    : list of [x1,y1,x2,y2]
        det_ids : optional list of track_id for Re-ID lookup
        returns : list of [x1,y1,x2,y2, track_id]
        """
        self.frame_count += 1
        predicted = [t.predict() for t in self.trackers]
        results = []

        if self.trackers and dets:
            n_d, n_t = len(dets), len(predicted)
            cost_mat = np.zeros((n_d, n_t), dtype=np.float32)
            for d,det in enumerate(dets):
                for t,pred in enumerate(predicted):
                    iou_cost = 1.0 - _iou_bb(det, pred)
                    # Re-ID cost (if embeddings available for this detection)
                    reid_cost = 0.5   # neutral default
                    if self.alpha > 0 and det_ids is not None:
                        did = det_ids[d]
                        tid_internal = self.trackers[t].id + 1
                        d_feat = self.reid_map.get(did)
                        t_feat = self.id_to_reid.get(tid_internal)
                        if d_feat is not None and t_feat is not None:
                            reid_cost = cosine_dist(d_feat, t_feat)
                    cost_mat[d,t] = self.alpha*reid_cost + (1-self.alpha)*iou_cost

            row_ind, col_ind = linear_sum_assignment(cost_mat)
            matched_d = set(); matched_t = set()
            for r,c in zip(row_ind, col_ind):
                if cost_mat[r,c] < (1 - self.iou_thresh):
                    self.trackers[c].update(dets[r])
                    matched_d.add(r); matched_t.add(c)
                    # Cache Re-ID feature for this track
                    if det_ids is not None and det_ids[r] in self.reid_map:
                        self.id_to_reid[self.trackers[c].id+1] = self.reid_map[det_ids[r]]

            for d,det in enumerate(dets):
                if d not in matched_d:
                    self.trackers.append(KalmanBoxTracker(det))
        else:
            for det in dets:
                self.trackers.append(KalmanBoxTracker(det))

        self.trackers = [t for t in self.trackers if t.time_since_update <= self.max_age]
        for t in self.trackers:
            if t.time_since_update==0 and t.hit_streak>=self.min_hits:
                results.append(t._state()+[t.id+1])
        return results

print("Running SORT (pure IoU) ...")
KalmanBoxTracker.count = 0
sort_tracker   = SORTWithReID(max_age=3, min_hits=2, iou_thresh=0.25,
                               reid_map=reid_map, alpha=0.0)   # pure IoU = baseline
sort_rows   = []; id_sw_sort = 0; prev_sort = set()

cap = cv2.VideoCapture(str(VIDEO_FILE)); frame_idx=0; t0=time.time()
while frame_idx < FRAMES_TO_RUN:
    ret, frame = cap.read()
    if not ret: break
    fd = TRACKS_DF[TRACKS_DF["frame"]==frame_idx]
    dets = fd[["x1","y1","x2","y2"]].values.tolist()
    dids = fd["track_id"].tolist()
    out  = sort_tracker.update(dets, dids)
    curr = {int(t[4]) for t in out}
    if frame_idx > 0: id_sw_sort += len(prev_sort - curr)
    prev_sort = curr
    for t in out:
        sort_rows.append({"frame":frame_idx,"track_id":int(t[4]),
                          "x1":round(t[0],1),"y1":round(t[1],1),"x2":round(t[2],1),"y2":round(t[3],1),
                          "cx":round((t[0]+t[2])/2,1),"cy":round((t[1]+t[3])/2,1),"width":round(t[2]-t[0],1)})
    if frame_idx%50==0: print(f"  Frame {frame_idx} | {time.time()-t0:.0f}s")
    frame_idx += 1
cap.release()
SORT_DF = pd.DataFrame(sort_rows)
SORT_DF.to_csv(STATS_OUT_DIR/f"{SELECTED_SEQ.name}_sort_tracks.csv",index=False)

# ── Run augmented SORT (Re-ID + IoU) for overlap comparison ─────────────────
print("\nRunning SORT+ReID ...")
KalmanBoxTracker.count = 0
sort_reid_tracker = SORTWithReID(max_age=3, min_hits=2, iou_thresh=0.25,
                                  reid_map=reid_map, alpha=REID_ALPHA if REID_AVAILABLE else 0.0)
reid_rows = []; id_sw_reid = 0; prev_reid = set()
cap = cv2.VideoCapture(str(VIDEO_FILE)); frame_idx=0; t0=time.time()
while frame_idx < FRAMES_TO_RUN:
    ret, frame = cap.read()
    if not ret: break
    fd   = TRACKS_DF[TRACKS_DF["frame"]==frame_idx]
    dets = fd[["x1","y1","x2","y2"]].values.tolist()
    dids = fd["track_id"].tolist()
    out  = sort_reid_tracker.update(dets, dids)
    curr = {int(t[4]) for t in out}
    if frame_idx > 0: id_sw_reid += len(prev_reid - curr)
    prev_reid = curr
    for t in out:
        reid_rows.append({"frame":frame_idx,"track_id":int(t[4]),
                          "x1":round(t[0],1),"y1":round(t[1],1),"x2":round(t[2],1),"y2":round(t[3],1),
                          "cx":round((t[0]+t[2])/2,1),"cy":round((t[1]+t[3])/2,1),"width":round(t[2]-t[0],1)})
    if frame_idx%50==0: print(f"  Frame {frame_idx} | {time.time()-t0:.0f}s")
    frame_idx += 1
cap.release()
SORT_REID_DF = pd.DataFrame(reid_rows)

print(f"\n{'='*55}")
print(f"  {'Tracker':<20}  {'Unique IDs':>10}  {'ID Switches':>11}")
print(f"{'='*55}")
print(f"  {'ByteTrack':<20}  {TRACKS_DF['track_id'].nunique():>10}  {'(see Cell 11)':>11}")
print(f"  {'SORT (IoU only)':<20}  {SORT_DF['track_id'].nunique():>10}  {id_sw_sort:>11}")
print(f"  {'SORT + ReID':<20}  {SORT_REID_DF['track_id'].nunique():>10}  {id_sw_reid:>11}")
print(f"{'='*55}")
print("Cell 10 complete.")

In [ ]:
# Quick peek at the GT file
with open(GT_FILE, 'r') as f:
    for _ in range(5):
        print(f.readline().rstrip())

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 11 — Evaluation Metrics & Comparison  (unchanged logic)
# ────────────────────────────────────────────────────────────────────────────────

def compute_mot_metrics(tracks_df, gt_file, label="Tracker"):
    if not gt_file.exists():
        print(f"GT not found: {gt_file}"); return None
    gt = pd.read_csv(
        gt_file,
        header=None,
        sep=',',   # explicit comma separator
        names=["frame", "id", "x", "y", "w", "h", "conf",
               "x_world", "y_world", "z_world"]
    )
    # Keep only annotated objects (conf = 1)
    gt = gt[gt["conf"] == 1].copy()
    gt["x2"]=(gt["x"]+gt["w"]); gt["y2"]=(gt["y"]+gt["h"])
    gt["cx"]=(gt["x"]+gt["w"]/2); gt["cy"]=(gt["y"]+gt["h"]/2)
    gt["frame"] -= 1
    gt = gt[gt["frame"] < FRAMES_TO_RUN]
    total_gt=0; total_det=0; total_matches=0; id_sw=0; prev_m={}
    for fid in range(FRAMES_TO_RUN):
        gf = gt[gt["frame"]==fid]; df = tracks_df[tracks_df["frame"]==fid]
        total_gt+=len(gf); total_det+=len(df)
        if gf.empty or df.empty: prev_m={}; continue
        curr_m={}; used=set()
        for _,g in gf.iterrows():
            dx=df["cx"]-g["cx"]; dy=df["cy"]-g["cy"]
            dist=np.sqrt(dx**2+dy**2); bi=dist.idxmin(); bd=dist[bi]
            thr=df.loc[bi,"width"]*1.5 if "width" in df.columns else 80
            if bd<thr and bi not in used:
                did=int(df.loc[bi,"track_id"]); curr_m[int(g["id"])]=did; used.add(bi)
                total_matches+=1
                if int(g["id"]) in prev_m and prev_m[int(g["id"])]!=did: id_sw+=1
        prev_m=curr_m
    prec=total_matches/total_det if total_det>0 else 0
    rec =total_matches/total_gt  if total_gt >0 else 0
    fp=total_det-total_matches; fn=total_gt-total_matches
    mota=1-(fn+fp+id_sw)/total_gt if total_gt>0 else 0
    idf1=2*total_matches/(total_gt+total_det) if (total_gt+total_det)>0 else 0
    return {"Tracker":label,"MOTA↑":round(mota*100,2),"IDF1↑":round(idf1*100,2),
            "Precision↑":round(prec*100,2),"Recall↑":round(rec*100,2),
            "ID Switches↓":id_sw,"GT":total_gt,"Dets":total_det,"Matches":total_matches}

m_byte = compute_mot_metrics(TRACKS_DF,   GT_FILE, "ByteTrack")
m_sort = compute_mot_metrics(SORT_DF,     GT_FILE, "SORT (IoU)")
m_reid = compute_mot_metrics(SORT_REID_DF,GT_FILE, "SORT+ReID")

all_metrics = [m for m in [m_byte, m_sort, m_reid] if m is not None]
if all_metrics:
    res_df = pd.DataFrame(all_metrics)
    res_df.to_csv(STATS_OUT_DIR/f"{SELECTED_SEQ.name}_evaluation.csv",index=False)
    print(res_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(12,5))
    mkeys = ["MOTA↑","IDF1↑","Precision↑","Recall↑"]
    x = np.arange(len(mkeys)); w = 0.25
    colors = ["#2563EB","#DC2626","#16A34A"]
    for i, (m, c) in enumerate(zip(all_metrics, colors)):
        bars = ax.bar(x + (i-1)*w, [m[k] for k in mkeys], w, label=m["Tracker"], color=c, alpha=0.85)
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(mkeys, fontsize=10)
    ax.set_ylim(0,115); ax.set_ylabel("Score (%)"); ax.legend()
    ax.set_title("Tracker Comparison — ByteTrack vs SORT vs SORT+ReID", fontsize=12, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(STATS_OUT_DIR/f"{SELECTED_SEQ.name}_comparison.png",dpi=150,bbox_inches="tight")
    plt.show()
else:
    print("GT not found — showing ID counts only.")
    print(f"ByteTrack IDs : {TRACKS_DF['track_id'].nunique()}")
    print(f"SORT IDs      : {SORT_DF['track_id'].nunique()}")
    print(f"SORT+ReID IDs : {SORT_REID_DF['track_id'].nunique()}")

# ── TrackEval pointer ─────────────────────────────────────────────────────────────────────────
# For proper HOTA/MOTA, use the bundled TrackEval:
#   python SportsMOT-main/codes/evaluation/TrackEval/scripts/run_mot_challenge.py \
#          --GT_FOLDER   <gt_dir>            \
#          --TRACKERS_FOLDER outputs/stats/  \
#          --BENCHMARK   TeamTrack

print("Cell 11 complete.")

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# CELL 12 — Local Streamlit App  (no ngrok — just run locally)
# ────────────────────────────────────────────────────────────────────────────────
# Run from your terminal (not inside the notebook):
#   pip install streamlit
#   streamlit run app.py
#
# The code below writes app.py to the working directory.

APP_CODE = '''
import streamlit as st
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib; matplotlib.use("Agg")
from pathlib import Path
from ultralytics import YOLO
import supervision as sv
import tempfile, time, warnings, os

st.set_page_config(page_title="Sports Tracking Analytics", page_icon="⚽", layout="wide")
st.title("⚽ Sports Player & Ball Tracking")
st.caption("CSE 429 · Computer Vision and Pattern Recognition · E-JUST")

st.sidebar.header("Configuration")
det_conf   = st.sidebar.slider("Detection confidence", 0.1, 0.9, 0.35, 0.05)
max_frames = st.sidebar.slider("Max frames", 50, 500, 150, 50)
show_ball  = st.sidebar.checkbox("Detect ball", value=True)

uploaded = st.file_uploader("Upload a sports video (MP4, AVI)", type=["mp4","avi","mov"])
if uploaded is None:
    st.info("Upload a video to start.")
    st.stop()

with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
    f.write(uploaded.read()); tmp_path = f.name

@st.cache_resource
def load_models():
    pm = YOLO("yolov8m.pt"); bm = YOLO("yolov8x.pt")
    tr = sv.ByteTrack(track_activation_threshold=0.35)
    return pm, bm, tr

player_model, ball_model, tracker = load_models()
tracker.reset()

cap = cv2.VideoCapture(tmp_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
to_run = min(max_frames, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

progress = st.progress(0); status = st.empty()
track_rows, ball_rows, frames_out = [], [], []
frame_idx = 0

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    while frame_idx < to_run:
        ret, frame = cap.read()
        if not ret: break
        res = player_model(frame, conf=det_conf, classes=[0], verbose=False)[0]
        dets = sv.Detections.from_ultralytics(res)
        tracked = tracker.update_with_detections(dets)
        for i in range(len(tracked)):
            x1,y1,x2,y2 = tracked.xyxy[i]
            track_rows.append({"frame":frame_idx,"track_id":int(tracked.tracker_id[i]),
                                "cx":float((x1+x2)/2),"cy":float((y1+y2)/2),"team":0})
        ball_pos = None
        if show_ball:
            br = ball_model(frame, conf=0.10, classes=[32], verbose=False)[0]
            if len(br.boxes) > 0:
                bx = br.boxes.xyxy[0].cpu().numpy()
                ball_pos = (int((bx[0]+bx[2])/2), int((bx[1]+bx[3])/2))
                ball_rows.append({"frame":frame_idx,"cx":ball_pos[0],"cy":ball_pos[1]})
        ann = frame.copy()
        ann = sv.BoxAnnotator(thickness=2).annotate(ann, tracked)
        ann = sv.LabelAnnotator(text_scale=0.4).annotate(ann, tracked,
              labels=[f"#{int(t)}" for t in tracked.tracker_id])
        if ball_pos: cv2.circle(ann, ball_pos, 12, (0,255,0), 3)
        frames_out.append(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        frame_idx += 1
        progress.progress(frame_idx / to_run)
        status.text(f"Processing {frame_idx}/{to_run} ...")

cap.release(); os.unlink(tmp_path); status.text("Done!")
tracks_df = pd.DataFrame(track_rows); ball_df = pd.DataFrame(ball_rows)

tab1, tab2, tab3 = st.tabs(["Annotated Frame", "Stats", "Heatmap"])
with tab1:
    mid = len(frames_out)//2
    st.image(frames_out[mid], caption=f"Frame {mid}", use_container_width=True)
with tab2:
    col1,col2,col3 = st.columns(3)
    col1.metric("Frames", frame_idx)
    col2.metric("Unique IDs", tracks_df["track_id"].nunique() if len(tracks_df) else 0)
    col3.metric("Ball detections", len(ball_df))
with tab3:
    if len(tracks_df) > 0:
        fig, ax = plt.subplots(figsize=(14,4))
        ax.set_facecolor("#2d8a4e")
        from scipy.stats import gaussian_kde
        xy = np.vstack([tracks_df["cx"], tracks_df["cy"]])
        if xy.shape[1] > 5:
            kde = gaussian_kde(xy, bw_method=0.15)
            xi = np.linspace(0,w,300); yi = np.linspace(0,h,100)
            Xi,Yi = np.meshgrid(xi,yi)
            Z = kde(np.vstack([Xi.ravel(),Yi.ravel()])).reshape(100,300)
            ax.imshow(Z,extent=[0,w,h,0],origin="upper",cmap="hot",alpha=0.6,aspect="auto")
        ax.axis("off"); ax.set_title("Player Heatmap", color="white")
        st.pyplot(fig)
'''

app_path = Path("app.py")
app_path.write_text(APP_CODE)
print(f"Streamlit app written to: {app_path.resolve()}")
print("\nTo run locally:")
print("  streamlit run app.py")
print("\nCell 12 complete — project deliverable ready.")